# Unsupervised Learning — Steel Plates Faults

Notebook Person 4: clustering dengan fitur numerik tanpa memakai label fault saat proses clustering.

**Tujuan:** preprocessing, membandingkan K-Means/Agglomerative/DBSCAN, evaluasi metrik, PCA, dan interpretasi cluster.

Jalankan dari root repository atau folder `notebooks/`. Dataset dan `src/data_pipeline.py` harus tersedia.

## 1. Import library dan konfigurasi

In [ ]:
from pathlib import Path
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
%matplotlib inline
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
cwd = Path.cwd().resolve()
if (cwd / "src" / "data_pipeline.py").exists():
    ROOT = cwd
elif (cwd.parent / "src" / "data_pipeline.py").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Tidak menemukan src/data_pipeline.py. Jalankan dari root repo atau folder notebooks.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data_pipeline import FEATURES, FAULT_TYPES, RAW
print("Project root:", ROOT)
print("Raw data path:", RAW)


**Penjelasan:** Mengimpor pustaka dan konfigurasi fitur/path dari pipeline project agar konsisten dengan notebook anggota tim lain.

## 2. Memuat data dan memisahkan fitur

In [ ]:
df = pd.read_csv(RAW)
df.columns = df.columns.str.strip()
if "K_Scratch" in df.columns and "K_Scatch" not in df.columns:
    df = df.rename(columns={"K_Scratch": "K_Scatch"})
missing_features = [c for c in FEATURES if c not in df.columns]
if missing_features:
    raise ValueError(f"Kolom fitur tidak ditemukan: {missing_features}")
X = df[FEATURES].copy()
available_fault_types = [c for c in FAULT_TYPES if c in df.columns]
print("Ukuran dataset:", df.shape)
print("Jumlah fitur clustering:", len(FEATURES))
print("Kolom label fault tersedia:", available_fault_types)
display(df.head())


**Penjelasan:** Clustering hanya memakai `FEATURES`. Label fault tidak dipakai untuk membentuk cluster; jika tersedia, label hanya boleh dipakai untuk interpretasi setelahnya.

## 3. Pemeriksaan awal data

In [ ]:
print("Informasi dataset:")
df.info()
print("Statistik deskriptif fitur:")
display(X.describe().T)
missing_counts = X.isna().sum().sort_values(ascending=False)
print("Missing values per fitur:")
display(missing_counts[missing_counts > 0])
print("Jumlah baris duplikat:", int(df.duplicated().sum()))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
missing_counts[missing_counts > 0].sort_values().plot(kind="barh", ax=axes[0], title="Missing values per fitur")
if (missing_counts > 0).sum() == 0:
    axes[0].text(0.5, 0.5, "Tidak ada missing value", ha="center", va="center")
    axes[0].set_axis_off()
X.std(numeric_only=True).sort_values().plot(kind="bar", ax=axes[1], title="Standar deviasi fitur")
axes[1].tick_params(axis="x", labelrotation=90)
plt.tight_layout()
plt.show()


**Penjelasan:** Pemeriksaan ini mengungkap missing value, duplikasi, statistik fitur, dan perbedaan skala yang penting untuk algoritma berbasis jarak.

## 4. Preprocessing

In [ ]:
X_numeric = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
all_empty = X_numeric.columns[X_numeric.isna().all()].tolist()
if all_empty:
    print("Fitur seluruhnya kosong dikeluarkan:", all_empty)
    X_numeric = X_numeric.drop(columns=all_empty)
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X_numeric), columns=X_numeric.columns, index=X_numeric.index)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=X_imputed.columns, index=X_imputed.index)
assert np.isfinite(X_scaled.to_numpy()).all(), "Masih ada nilai non-finite."
print("Jumlah baris:", len(X_scaled))
print("Jumlah fitur setelah preprocessing:", X_scaled.shape[1])
display(X_scaled.head())


**Penjelasan:** Nilai non-numerik/tak hingga menjadi missing value, lalu diisi median. StandardScaler menyamakan skala fitur. Fitur yang seluruhnya kosong dikeluarkan.

## 5. K-Means — Elbow Method

In [ ]:
k_values = list(range(2, min(10, len(X_scaled) - 1) + 1))
inertias = []
for k in k_values:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)
plt.figure(figsize=(8, 5))
plt.plot(k_values, inertias, marker="o")
plt.xlabel("Jumlah cluster (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method untuk K-Means")
plt.xticks(k_values)
plt.show()


**Penjelasan:** Inertia mengukur jumlah kuadrat jarak titik ke centroid. Kandidat k dapat dipilih di sekitar titik saat penurunan inertia mulai melambat.

## 6. K-Means — Silhouette Score

In [ ]:
silhouette_by_k = {}
for k in k_values:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels_k = model.fit_predict(X_scaled)
    if len(np.unique(labels_k)) > 1:
        silhouette_by_k[k] = silhouette_score(X_scaled, labels_k)
silhouette_k_df = pd.DataFrame({"k": list(silhouette_by_k.keys()), "silhouette": list(silhouette_by_k.values())})
display(silhouette_k_df)
plt.figure(figsize=(8, 5))
plt.plot(silhouette_k_df["k"], silhouette_k_df["silhouette"], marker="o")
plt.xlabel("Jumlah cluster (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score untuk K-Means")
plt.xticks(k_values)
plt.show()
if not silhouette_by_k:
    raise ValueError("Tidak ada nilai k yang menghasilkan beberapa cluster.")
optimal_k = max(silhouette_by_k, key=silhouette_by_k.get)
print("k terpilih berdasarkan Silhouette Score:", optimal_k)


**Penjelasan:** Silhouette lebih tinggi umumnya menunjukkan pemisahan cluster yang lebih baik. k dipilih berdasarkan skor tertinggi pada rentang uji, lalu ditinjau bersama Elbow dan konteks masalah.

## 7. K-Means dan profil cluster

In [ ]:
kmeans = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)
labels_kmeans = kmeans.fit_predict(X_scaled)
clustered = X_imputed.copy()
clustered["KMeans_Cluster"] = labels_kmeans
print("Jumlah data per cluster K-Means:")
display(clustered["KMeans_Cluster"].value_counts().sort_index().rename("jumlah_data"))
print("Rata-rata fitur pada skala asli per cluster:")
kmeans_profile = clustered.groupby("KMeans_Cluster")[X_imputed.columns].mean()
display(kmeans_profile)
clustered["KMeans_Cluster"].value_counts().sort_index().plot(kind="bar", figsize=(8, 4), title="Distribusi Cluster K-Means")
plt.xlabel("Cluster")
plt.ylabel("Jumlah data")
plt.show()


**Penjelasan:** Profil rata-rata memakai fitur yang sudah diimputasi tetapi belum distandardisasi agar satuan fitur lebih mudah ditafsirkan. Cluster bukan otomatis jenis kerusakan tertentu.

## 8. Agglomerative Clustering

In [ ]:
agglomerative = AgglomerativeClustering(n_clusters=optimal_k, linkage="ward")
labels_agg = agglomerative.fit_predict(X_scaled)
clustered["Agglomerative_Cluster"] = labels_agg
print("Jumlah data per cluster Agglomerative:")
display(clustered["Agglomerative_Cluster"].value_counts().sort_index().rename("jumlah_data"))
agg_profile = clustered.groupby("Agglomerative_Cluster")[X_imputed.columns].mean()
display(agg_profile)


**Penjelasan:** Agglomerative menggabungkan kelompok secara bertahap. Linkage Ward berusaha menjaga kenaikan variasi dalam cluster tetap kecil.

## 9. DBSCAN — pencarian parameter

In [ ]:
dbscan_candidates = []
for eps in [1.5, 2.0, 2.5, 3.0]:
    for min_samples in [5, 10]:
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_scaled)
        non_noise = labels != -1
        n_clusters = len(set(labels[non_noise]))
        score = np.nan
        if n_clusters >= 2 and non_noise.sum() > n_clusters:
            score = silhouette_score(X_scaled.loc[non_noise], labels[non_noise])
        dbscan_candidates.append({"eps": eps, "min_samples": min_samples,
            "jumlah_cluster_non_noise": n_clusters,
            "noise_percent": 100 * np.mean(labels == -1),
            "silhouette_non_noise": score})
dbscan_search = pd.DataFrame(dbscan_candidates).sort_values("silhouette_non_noise", ascending=False, na_position="last")
display(dbscan_search)
valid = dbscan_search.dropna(subset=["silhouette_non_noise"])
if not valid.empty:
    best = valid.iloc[0]
    best_eps, best_min_samples = float(best["eps"]), int(best["min_samples"])
else:
    best_eps, best_min_samples = 2.0, 5
labels_dbscan = DBSCAN(eps=best_eps, min_samples=best_min_samples).fit_predict(X_scaled)
clustered["DBSCAN_Cluster"] = labels_dbscan
print(f"Parameter DBSCAN terpilih: eps={best_eps}, min_samples={best_min_samples}")
display(clustered["DBSCAN_Cluster"].value_counts().sort_index().rename("jumlah_data; -1 berarti noise"))


**Penjelasan:** DBSCAN memakai radius `eps` dan `min_samples`, bukan jumlah cluster yang ditentukan di awal. Label -1 berarti noise/outlier. Pencarian parameter ini bersifat awal.

## 10. Perbandingan metrik

In [ ]:
def evaluate_clustering(X_data, labels, ignore_noise=False):
    labels = np.asarray(labels)
    mask = labels != -1 if ignore_noise else np.ones(len(labels), dtype=bool)
    X_eval = np.asarray(X_data)[mask]
    labels_eval = labels[mask]
    n_clusters = len(np.unique(labels_eval))
    result = {"jumlah_cluster": n_clusters, "silhouette": np.nan,
              "davies_bouldin": np.nan, "calinski_harabasz": np.nan,
              "noise_percent": 100 * np.mean(labels == -1) if ignore_noise else 0.0}
    if n_clusters >= 2 and len(labels_eval) > n_clusters:
        result["silhouette"] = silhouette_score(X_eval, labels_eval)
        result["davies_bouldin"] = davies_bouldin_score(X_eval, labels_eval)
        result["calinski_harabasz"] = calinski_harabasz_score(X_eval, labels_eval)
    return result
comparison = pd.DataFrame({
    "K-Means": evaluate_clustering(X_scaled, labels_kmeans),
    "Agglomerative": evaluate_clustering(X_scaled, labels_agg),
    "DBSCAN": evaluate_clustering(X_scaled, labels_dbscan, ignore_noise=True)
}).T
display(comparison.round(4))


**Penjelasan:** Silhouette lebih tinggi umumnya lebih baik; Davies–Bouldin lebih rendah umumnya lebih baik; Calinski–Harabasz lebih tinggi umumnya lebih baik. Metrik DBSCAN hanya dihitung pada titik non-noise, sehingga perbandingan tidak sepenuhnya setara.

## 11. PCA dan visualisasi 2D

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
components_80 = int(np.argmax(cumulative_variance >= 0.80) + 1)
print("Jumlah komponen untuk mencapai >=80% variasi:", components_80)
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker="o")
plt.axhline(0.80, linestyle="--", label="80% variasi")
plt.axvline(components_80, linestyle="--", label=f"{components_80} komponen")
plt.xlabel("Jumlah komponen")
plt.ylabel("Variasi kumulatif yang dijelaskan")
plt.title("PCA — Cumulative Explained Variance")
plt.legend()
plt.show()
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_scaled)
print("Variasi yang dijelaskan dua komponen:", pca_2d.explained_variance_ratio_.sum())
label_sets = {"K-Means": labels_kmeans, "Agglomerative": labels_agg, "DBSCAN": labels_dbscan}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, labels) in zip(axes, label_sets.items()):
    ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=labels, cmap="tab10", s=18, alpha=0.75)
    ax.set_title(f"PCA 2D — {name}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()


**Penjelasan:** PCA mereduksi dimensi dengan mempertahankan variasi sebanyak mungkin. `components_80` menunjukkan jumlah komponen minimum untuk menjelaskan setidaknya 80% variasi. Plot 2D hanya menampilkan dua komponen pertama.

## 12. Interpretasi dan kesimpulan

In [ ]:
print("Ringkasan evaluasi:")
display(comparison.round(4))
print("Distribusi cluster:")
for name, labels in label_sets.items():
    print(chr(10), name)
    display(pd.Series(labels, name="cluster").value_counts().sort_index().rename("jumlah_data"))
profile_data = X_imputed.copy()
profile_data["cluster"] = labels_kmeans
means = profile_data.groupby("cluster")[X_imputed.columns].mean()
feature_ranges = (means.max(axis=0) - means.min(axis=0)).sort_values(ascending=False)
print("10 fitur dengan rentang rata-rata terbesar antar-cluster K-Means:")
display(feature_ranges.head(10).rename("rentang_rata_rata"))
valid_silhouette = comparison["silhouette"].dropna()
if not valid_silhouette.empty:
    print("Silhouette tertinggi pada evaluasi ini:", valid_silhouette.idxmax())
else:
    print("Silhouette tidak dapat dihitung untuk hasil yang tersedia.")
print("Panduan: periksa fitur pembeda, distribusi cluster, dan proporsi noise DBSCAN. Jangan mengartikan cluster sebagai jenis fault tanpa validasi. Label fault hanya dianalisis setelah clustering.")


**Penjelasan:** Gunakan metrik, distribusi cluster, dan profil fitur untuk membuat kesimpulan berdasarkan hasil aktual.

**Isi kesimpulan setelah menjalankan notebook:**
- Metode paling sesuai berdasarkan metrik dan interpretasi: …
- Jumlah cluster dan pola utama: …
- Fitur pembeda utama: …
- Proporsi noise DBSCAN dan implikasinya: …
- Keterbatasan: hasil dipengaruhi preprocessing, skala, parameter, dan PCA; cluster tidak otomatis sama dengan kategori fault.